# PulseForge AI — Milestone M2: Audio Exploration & Feature Preprocessing

This notebook demonstrates:
1. Loading authentic normal and anomalous fan audio recordings from the MIMII dataset.
2. Applying safety validation, channel selection (mono downmix/first channel), and standardized resampling.
3. Visualizing raw waveforms and log-Mel power spectrograms.
4. Framing signals into overlapping 1.0-second segments.
5. Extracting fixed-shape log-Mel features `(num_segments, 128, 32)`.
6. Fitting `FeatureScaler` strictly on normal training data and verifying normalization.

In [ ]:
import sys
import csv
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import get_default_config
from src.audio_io import load_audio, get_audio_metadata
from src.preprocessing import preprocess_audio, frame_signal
from src.features import (
    extract_features,
    extract_log_mel_spectrogram,
    FeatureScaler,
    compute_waveform_plot_data,
    compute_spectrogram_plot_data,
)

cfg = get_default_config()
print("Project root:", PROJECT_ROOT)
print("Sample rate:", cfg.audio.sample_rate)
print("Mel bins:", cfg.features.n_mels)

## 1. Load Dataset Manifest
Inspect the reproducible manifest generated in Milestone M1.

In [ ]:
manifest_path = PROJECT_ROOT / "data" / "manifests" / "fan_manifest.csv"
with open(manifest_path, "r", encoding="utf-8") as f:
    records = list(csv.DictReader(f))

normal_samples = [r for r in records if r["label"] == "normal"]
abnormal_samples = [r for r in records if r["label"] == "abnormal"]

print(f"Total manifest records: {len(records)}")
print(f"Normal samples: {len(normal_samples)}, Abnormal samples: {len(abnormal_samples)}")

## 2. Load Normal and Abnormal Audio Signals

In [ ]:
normal_rec = normal_samples[0]
abnormal_rec = abnormal_samples[0]

audio_normal = load_audio(PROJECT_ROOT / normal_rec["relative_path"], config=cfg.audio)
audio_abnormal = load_audio(PROJECT_ROOT / abnormal_rec["relative_path"], config=cfg.audio)

print(f"Normal clip: {normal_rec['recording_id']}, shape={audio_normal.data.shape}, sr={audio_normal.sample_rate} Hz, duration={audio_normal.duration_seconds}s")
print(f"Abnormal clip: {abnormal_rec['recording_id']}, shape={audio_abnormal.data.shape}, sr={audio_abnormal.sample_rate} Hz, duration={audio_abnormal.duration_seconds}s")

## 3. Waveform and Log-Mel Spectrogram Diagnostics

In [ ]:
def plot_diagnostics(audio, title):
    time_wave, amp = compute_waveform_plot_data(audio.data, audio.sample_rate)
    mel_db, time_spec, freqs = compute_spectrogram_plot_data(audio.data, audio.sample_rate, config=cfg.features)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True, constrained_layout=True)
    ax1.plot(time_wave, amp, color="#2563EB", linewidth=0.6, alpha=0.85)
    ax1.set_ylabel("Amplitude")
    ax1.set_title(title)
    ax1.grid(True, linestyle="--", alpha=0.5)

    img = ax2.imshow(
        mel_db,
        origin="lower",
        aspect="auto",
        extent=[time_spec[0], time_spec[-1], freqs[0], freqs[-1]],
        cmap="magma",
    )
    ax2.set_xlabel("Time (seconds)")
    ax2.set_ylabel("Frequency (Hz)")
    cbar = fig.colorbar(img, ax=ax2, orientation="horizontal", pad=0.15, shrink=0.6)
    cbar.set_label("Log-Mel Energy (dB)")
    plt.show()

plot_diagnostics(audio_normal, "Normal Fan Signature")
plot_diagnostics(audio_abnormal, "Anomalous Fan Signature")

## 4. Sliding Window Segmentation & Feature Tensor Extraction
Audio clips (10s) are segmented into 1-second windows with 0.5-second hop, yielding 19 segments per recording. Each segment is transformed into a log-Mel spectrogram of shape `(128, 32)`.

In [ ]:
features_norm = extract_features(audio_normal, config=cfg.features)
features_abn = extract_features(audio_abnormal, config=cfg.features)

print(f"Normal feature tensor shape: {features_norm.features.shape} (N_segments, N_mels, N_frames)")
print(f"Abnormal feature tensor shape: {features_abn.features.shape}")
print(f"Flattened vector per segment: {features_norm.flattened_dim}")
assert features_norm.features.shape == (19, 128, 32)
assert np.all(np.isfinite(features_norm.features))

## 5. Fit FeatureScaler on Normal Training Data
Normalization statistics (mean and standard deviation per Mel frequency bin) are fitted strictly on the normal training split to prevent any data leakage.

In [ ]:
train_records = [r for r in records if r["split"] == "train"]
print(f"Fitting scaler on {len(train_records)} normal training files...")

train_features = []
for r in train_records:
    sig = load_audio(PROJECT_ROOT / r["relative_path"], config=cfg.audio)
    feat = extract_features(sig, config=cfg.features)
    train_features.append(feat.features)

all_train = np.concatenate(train_features, axis=0)
print(f"Total training segments: {all_train.shape[0]}, segment shape: {all_train.shape[1:]}")

scaler = FeatureScaler()
scaler.fit(all_train)

# Verify normalization
normed_train = scaler.transform(all_train)
print(f"Normalized train mean: {normed_train.mean():.4f} (target ~ 0.0)")
print(f"Normalized train std:  {normed_train.std():.4f} (target ~ 1.0)")

# Save scaler parameters
scaler_out = PROJECT_ROOT / "model_artifacts" / "scaler_params.json"
scaler.save(scaler_out)
print("FeatureScaler successfully saved to:", scaler_out)